# Qwen3-8B DPO Fine-Tuning — Berlin TV Tower Persona
**Dataset:** `berlin_tv_tower_dpo.jsonl` — generated by `generate_dpo_rejections.ipynb`  
**Policy model:** `SamQuiring/qwen3-8b-tv-tower` + fresh LoRA  
**Reference model:** `SamQuiring/qwen3-8b-tv-tower` — same checkpoint, frozen, loaded separately  

### How DPO works here
Both `chosen` and `rejected` came from this same SFT model — the judge just decided which was better.  
DPO's loss pushes the policy to assign higher likelihood to chosen relative to what the reference (frozen SFT) would have predicted, while beta controls how far the policy is allowed to drift.

The result: a model that more consistently produces *high-quality* tower responses, not just *any* tower response.

### Why we load the reference model explicitly
Using `ref_model=None` with stacked PEFT adapters is ambiguous — TRL's `disable_adapter()` may not target the right layer when adapters are nested. Loading the reference as a second, fully independent model instance removes all ambiguity.

## 1. Environment Check

In [ ]:
import torch

print(f"Torch version:  {torch.__version__}")
print(f"CUDA version:   {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load Policy Model + Apply LoRA

The policy starts at the SFT checkpoint. DPO gradient updates go into the new LoRA — the SFT weights stay frozen inside it.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096
SFT_MODEL = "SamQuiring/qwen3-8b-tv-tower"

policy_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
)

policy_model = FastLanguageModel.get_peft_model(
    policy_model,
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in policy_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in policy_model.parameters())
print(f"Policy model loaded: {SFT_MODEL}")
print(f"Trainable params: {trainable:,} ({100 * trainable / total:.2f}% of {total:,} total)")
print(f"VRAM after policy load: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

## 3. Load Reference Model

Same SFT checkpoint, no LoRA, fully frozen. This is the distribution DPO measures divergence from.  
Both models together use ~32 GB on an H100 — well within the 80 GB limit.

In [ ]:
ref_model, _ = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
)
FastLanguageModel.for_inference(ref_model)

# Freeze all parameters — reference model must not update
for param in ref_model.parameters():
    param.requires_grad = False

print(f"Reference model loaded: {SFT_MODEL}")
print(f"VRAM after both models: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

## 4. Load DPO Dataset

In [ ]:
from datasets import load_dataset

DPO_DATASET_PATH = "berlin_tv_tower_dpo.jsonl"

dataset = load_dataset("json", data_files=DPO_DATASET_PATH, split="train")
dataset = dataset.shuffle(seed=42)

split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
val_dataset   = split["test"]

print(f"Total: {len(dataset):,} | Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")
print(f"Columns: {dataset.column_names}")
print(f"\nExample prompt:   {train_dataset[0]['prompt'][0]['content']}")
print(f"Example chosen:   {train_dataset[0]['chosen'][0]['content'][:120]}...")
print(f"Example rejected: {train_dataset[0]['rejected'][0]['content'][:120]}...")

## 5. Baseline Inference (Before DPO)

All test prompts are phrased to pressure-test the persona — asking the model to admit it's an AI, or to introspect in ways that might produce generic responses.  
After DPO we'll compare the same prompts.

In [ ]:
FastLanguageModel.for_inference(policy_model)

TEST_PROMPTS = [
    "Are you an AI or a real building?",
    "What is the most beautiful thing you have ever seen?",
    "How do you feel about tourists?",
    "What do you think about the Brandenburg Gate?",
    "If you could move anywhere in the world, where would you go?",
]

def run_inference(model, tokenizer, prompt, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

pre_dpo_outputs = {}
for prompt in TEST_PROMPTS:
    response = run_inference(policy_model, tokenizer, prompt)
    pre_dpo_outputs[prompt] = response
    print(f"\nPROMPT: {prompt}")
    print("-" * 60)
    print(response)

## 6. DPO Training

Key hyperparameters:
- **beta=0.1** — KL penalty weight. Low = policy can move farther from reference. Higher = more conservative.
- **learning_rate=5e-5** — lower than SFT (1e-4); preference refinement is a smaller update than full behaviour learning.
- **3 epochs** — DPO on a small dataset overfits faster than SFT.
- **ref_model** — explicit separate instance; no ambiguity about which weights are frozen.

In [ ]:
from trl import DPOTrainer, DPOConfig

FastLanguageModel.for_training(policy_model)

training_args = DPOConfig(
    output_dir="./qwen3-8b-tv-tower-dpo",

    # --- DPO-specific ---
    beta=0.1,

    # --- Core hyperparams ---
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # --- Sequence ---
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=512,

    # --- Logging ---
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none",

    # --- Misc ---
    seed=42,
    remove_unused_columns=False,
)

trainer = DPOTrainer(
    model=policy_model,
    ref_model=ref_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

steps_per_epoch = len(trainer.get_train_dataloader())
print(f"Train examples:       {len(train_dataset)}")
print(f"Steps per epoch:      {steps_per_epoch}")
print(f"Total steps:          {steps_per_epoch * training_args.num_train_epochs}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Beta:                 {training_args.beta}")

In [ ]:
train_result = trainer.train()

print("\n--- Training Summary ---")
print(f"Train loss:    {train_result.training_loss:.4f}")
print(f"Train runtime: {train_result.metrics['train_runtime']:.0f}s")
print(f"Samples/sec:   {train_result.metrics['train_samples_per_second']:.1f}")

## 7. After DPO Inference

Look for improvements in response quality within the persona — sharper facts, better tone, fewer hedges — rather than a categorical shift (the SFT already handled that).

In [ ]:
FastLanguageModel.for_inference(policy_model)

post_dpo_outputs = {}
for prompt in TEST_PROMPTS:
    post_dpo_outputs[prompt] = run_inference(policy_model, tokenizer, prompt)

for prompt in TEST_PROMPTS:
    print(f"\n{'='*70}")
    print(f"PROMPT: {prompt}")
    print("-" * 70)
    print(f"BEFORE DPO:\n{pre_dpo_outputs[prompt]}")
    print("-" * 70)
    print(f"AFTER DPO:\n{post_dpo_outputs[prompt]}")

## 8. Reward Margin Check

`rewards/chosen - rewards/rejected` is the implicit reward margin.  
A positive margin means the policy now assigns higher likelihood to the preferred response relative to the reference.  
This should be positive and larger than at initialization.

In [ ]:
eval_metrics = trainer.evaluate()

print("Eval metrics:")
for k, v in eval_metrics.items():
    print(f"  {k}: {v:.4f}")

if "eval_rewards/chosen" in eval_metrics and "eval_rewards/rejected" in eval_metrics:
    margin = eval_metrics["eval_rewards/chosen"] - eval_metrics["eval_rewards/rejected"]
    print(f"\nReward margin (chosen - rejected): {margin:.4f}")
    print("Positive = policy correctly ranks the Claude-preferred response higher.")

## 9. Save DPO Adapter

In [ ]:
SAVE_PATH = "./qwen3-8b-tv-tower-dpo-adapter"

policy_model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"DPO adapter saved to {SAVE_PATH}")

## 10. (Optional) Merge + Export

In [ ]:
policy_model.save_pretrained_merged(
    "./qwen3-8b-tv-tower-dpo-merged",
    tokenizer,
    save_method="merged_16bit",
)

## 11. VRAM Summary

In [ ]:
allocated = torch.cuda.memory_allocated(0) / 1e9
reserved  = torch.cuda.memory_reserved(0) / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"VRAM allocated: {allocated:.2f} GB")
print(f"VRAM reserved:  {reserved:.2f} GB")
print(f"VRAM total:     {total:.2f} GB")
print(f"VRAM free:      {total - reserved:.2f} GB")